# Рад са моделима породице Meta 

## Увод 

Ова лекција ће обухватити: 

- Истраживање два главна модела породице Meta - Llama 3.1 и Llama 3.2, плус тренутно доступних алтернатива коришћених у примерима кода (Llama 3.3 и Llama 4) 
- Разумевање случајева употребе и сценарија за сваки модел 
- Пример кода који показује јединствене карактеристике сваког модела 


## Породица модела Meta 

У овој ће лекцији бити истражена 2 модела из породице Meta или „Llama Herd“ - Llama 3.1 и Llama 3.2. Извршни примери кода користе Llama 3.3 и Llama 4, који су тренутно доступне алтернативе у каталогу Microsoft Foundry Models. 

Ови модели долазе у различитим варијантама и доступни су у [каталогу Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Белешка:** GitHub Models се повлачи крајем јула 2026. Више детаља о коришћењу [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) за прототипизацију са AI моделима.

> **Белешка:** Конкретне верзије модела према којима је ова лекција првобитно написана (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) су од тада повучене из каталога Microsoft Foundry Models. Примери кода испод користе `Llama-3.3-70B-Instruct` и `Llama-4-Scout-17B-16E-Instruct` као тренутно доступне алтернативе у каталогу Microsoft Foundry Models, па се и даље могу извршавати.

Варијанте модела (као што је првобитно обухваћено у овој лекцији): 
- Llama 3.1 - 70B Instruct (повучен) 
- Llama 3.1 - 405B Instruct (повучен) 
- Llama 3.2 - 11B Vision Instruct (повучен) 
- Llama 3.2 - 90B Vision Instruct (повучен) 

Тренутно доступне алтернативе у примерима кода: 
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`) 
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`) 

*Белешка: Варијанте Llama 3 описане горе задржане су из историјских разлога и више нису доступне у Microsoft Foundry Models.*



## Llama 3.1 

Са 405 милијарди параметара, Llama 3.1 спада у категорију отворених LLM мрежа. 

Овај модел је надоградња у односу на претходно издање Llama 3 и нуди: 

- Веће контекстуално окно - 128k токена уместо 8k токена 
- Већи максималан број излазних токена - 4096 уместо 2048 
- Боља подршка за више језика - због повећања броја токена за тренинг 

Ово омогућава Llama 3.1 да рукује сложенијим случајевима коришћења приликом изградње GenAI апликација, укључујући: 
- Нативно позивање функција - могућност позивања спољних алата и функција ван LLM процеса
- Боља RAG перформанса - због већег контекстуалног окна 
- Синтетичко генерисање података - могућност креирања ефикасних података за задатке као што је дотеривање модела 



### Позивање нативних функција 

Llama 3.1 је фино подешен да буде ефикаснији у позивању функција или алата. Такође има два уграђена алата које модел може препознати као потребне за коришћење на основу упита од корисника. Ови алати су: 

- **Brave Search** - Може се користити за добијање најновијих информација као што је време путем претраге интернета 
- **Wolfram Alpha** - Може се користити за сложеније математичке прорачуне тако да није потребно писати своје функције. 

Такође можете направити своје прилагођене алате које LLM може позивати. 

У примеру кода испод: 

- Дефинишемо доступне алате (brave_search, wolfram_alpha) у системском упиту. 
- Шаљемо упит корисника који пита о времену у одређеном граду. 
- LLM ће одговорити позивом алата Brave Search који ће изгледати овако `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Напомена: Овај пример само прави позив алату, ако желите да добијете резултате, потребно је да направите бесплатан налог на Brave API страници и дефинишете саму функцију` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Ллама 3.2

Иако је ЛЛМ, једно ограничење које Ллама 3.1 има је мултимодалност. То јест, способност коришћења различитих типова уноса као што су слике као упутства и пружање одговора. Ова способност је једна од главних карактеристика Ллама 3.2. Ове карактеристике такође укључују:

- Мултимодалност - има способност обраде како текстуалних тако и сликовних упутстава
- Варијанте мале до средње величине (11B и 90B) - омогућавају флексибилне опције распоређивања,
- Текстуалне варијанте (1B и 3B) - омогућавају да се модел користи на периферним / мобилним уређајима и обезбеђују ниску латенцију

Подршка за мултимодалност представља велики корак у свету open source модела. Пример кода испод узима и слику и текстуално упутство да добије анализу слике. Пошто је Ллама 3.2 90B Vision Instruct повучен, пример користи `Llama-4-Scout-17B-16E-Instruct`, тренутно доступну нативно мултимодалну алтернативу.

### Мултимодална подршка са Ллама 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Учење овде не престаје, наставите путовање

Након завршетка овог часа, погледајте нашу [Колекцију за учење генеративне вештачке интелигенције](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) како бисте наставили да унапређујете своје знање о генеративној вештачкој интелигенцији!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
